# Gate 0: is the detector orientation-sensitive?

Drives `src/rotation.py` over one owner's half of the matrix. Every checkpoint that already
exists is scored against `rot90(x_test, k)` for k = 0, 1, 2, 3 - **no training, no cache
rebuild, seconds per checkpoint** - and the answer decides whether the D4 augmentation
ablation is worth 28 GPU-runs.

The question is *not* whether a generator fingerprint survives rotation. It does, provably:
k x 90 deg is an exact permutation of the pixel grid, the radially averaged spectrum is
invariant under it to 1e-12, and 128 % 8 == 0 so the JPEG 8x8 lattice maps onto itself. Any
spectral test of that returns "invariant" by construction and carries no information.

The question is whether the **network** can use the rotated signal. A CNN is equivariant to
translation and to nothing else, so an oriented filter bank has to learn each orientation
separately - and if it has, D4 is 8x effective data for free.

| Result | Meaning |
|---|---|
| rotated ~ upright | Already orientation-invariant. D4 buys nothing. **Stop, and report the negative finding.** |
| rotated collapses toward 0.5 | ~7/8 of capacity is spent relearning one texture. **Run Gate 1.** |
| tnr falls further than tpr | The **real** class carries the orientation cue - the one mechanism by which D4 could move the *cross-generator* number, not just the diagonal. |

Output: `results/figures/rotation_sensitivity_<owner>.json`, a few KB, committed through git
exactly like `metrics.json` - so the two accounts' halves merge with no large transfer.

## Setup

Set `OWNER` in the next cell and nothing else. Same paths as `01_run_matrix.ipynb`: the cache
is read from the shared root, the checkpoints from **this account's own** Drive backup.

In [ ]:
# Re-run this after any runtime restart.
import os, sys, json, glob
from pathlib import Path

# A CPU runtime will run this - slowly, and `--device cuda` would fail outright. The check is
# cheap and the fallback message says what to do, because on a CPU runtime nvidia-smi is not
# installed at all and the bare "command not found" says nothing.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU on this runtime -> Runtime > Change runtime type > T4 GPU > Save, then re-run this cell (the restart unmounts Drive)"

from google.colab import drive
drive.mount('/content/drive')

# Who is running this notebook. Only this account's 28 checkpoints have weights here; the
# other 28 are skipped, which is expected rather than an error.
OWNER = "ido"

# Read root, shared between both accounts - identical bytes, nothing copied.
DRIVE = "/content/drive/MyDrive/university/deep_learning"
CACHE = f"{DRIVE}/cache"

# Write root, this account's OWN My Drive. Deliberately a different tree from CACHE so a
# view-only share cannot break it - see 01_run_matrix.ipynb, setup cell.
BACKUP = f"/content/drive/MyDrive/deep_learning_results/{OWNER}"

# Repo-relative: metrics.json comes from git and lists all 56 runs, so the loop sees the
# whole matrix and skips whatever has no local weights.
RESULTS = "results/runs"

assert os.path.isdir(DRIVE), (
    f"not found: {DRIVE}\nCheck it is mounted on THIS account: `ls /content/drive/MyDrive`."
)

print(f"owner   {OWNER}")
print("cache  <-", CACHE, "" if os.path.isdir(CACHE) else "  <- MISSING")
print("ckpts  <-", f"{BACKUP}/runs")

## Get the code

In [ ]:
# Idempotent: clones on the first run, pulls on every later one.
![ -d /content/deep-learning ] || git clone -q https://github.com/noa-keter/deep-learning.git /content/deep-learning
%cd /content/deep-learning
!git pull --ff-only
!git log --oneline -1

## Gate: are this account's checkpoints actually there?

`*.pt` is gitignored and `/content` is wiped on a runtime recycle, so the only surviving
weights are the Drive backup written by `01_run_matrix.ipynb` cell 17. Twenty-eight is a full
half of the matrix; anything less means part of the answer will be missing and the summary
will quietly average over fewer runs.

In [ ]:
found = sorted(glob.glob(f"{BACKUP}/runs/*/*/seed*/model.pt"))
metrics = sorted(glob.glob(f"{RESULTS}/*/*/seed*/metrics.json"))

by_strategy = {}
for path in found:
    by_strategy.setdefault(Path(path).parts[-4], []).append(path)

print(f"{len(metrics)} metrics.json from git (all 56 expected)")
print(f"{len(found)} checkpoints under {BACKUP}/runs (28 expected for one account)")
for strategy, paths in sorted(by_strategy.items()):
    print(f"  {strategy:<13}{len(paths):>3}")

if not found:
    print("\nNOTHING FOUND. Is OWNER right, and is Drive mounted on THIS account?")
elif len(found) < 28:
    print(f"\nOnly {len(found)} of 28 - the summary below will cover part of this half.")

## Run it

One `python -m src.rotation` subprocess, exactly as the matrix is driven: VRAM is fully
released at the end and a failure leaves the notebook alive to report it.

**The first thing to read in the output is the k=0 reproduction check.** `load_arm` is
seed-deterministic and this reuses `train._predict`, so the upright pass reproduces each run's
recorded `in_domain` and `off_domain_mean` - and the check reports how many **test rows**
disagree, allowing up to 4 per checkpoint before it raises.

The budget is not zero on purpose. `_predict` runs under fp16 autocast, so a score sitting
within rounding distance of the decision threshold can land on either side depending on which
GPU Colab handed out - observed in practice as one flipped row in 112,000 predictions. A wrong
arm, seed or cache moves whole percent instead, i.e. tens of rows out of 4,000, so the two are
three orders of magnitude apart. **1 or 2 rows is rounding; anything near the budget is not,
and nothing rotated below it should be believed.**

In [ ]:
gate0_cmd = (
    f'python -m src.rotation --cache-dir "{CACHE}" --ckpt-root "{BACKUP}/runs"'
    f' --owner {OWNER} --results-dir "{RESULTS}"'
)
!{gate0_cmd}

## Both halves together

Each account commits its own `rotation_sensitivity_<owner>.json`. Once both are on `main`,
this cell merges whatever is present and re-prints the tables and the verdict over all four
arms - the per-strategy table is the one that matters, because the mechanism is arm-specific:
`rescale` is where real images are aspect-distorted and generated ones are not, so an
orientation-signed real-class cue shows up there first.

In [ ]:
from src.rotation import format_report, verdict

rows = []
for path in sorted(glob.glob("results/figures/rotation_sensitivity_*.json")):
    part = json.loads(Path(path).read_text())
    rows += part
    print(f"{Path(path).name:<40}{len(part) // 4:>3} checkpoints")

arms = sorted({r["strategy"] for r in rows})
print(f"\n{len(rows) // 4} checkpoints over {len(arms)} arms: {', '.join(arms)}\n")
print(format_report(rows))
print("\nVERDICT:", verdict(rows))

## Hand off

The JSON goes through git, like `metrics.json` - it is a few KB and it is what merges the two
halves. Nothing goes to Drive from this notebook: no weights are produced and nothing here is
expensive to recompute.

If the verdict says **RUN Gate 1**, the change is in `src/train.py` beside the existing flip at
`FLIP_PROBABILITY` - and three rules from `_kb/state/PROJECT_STATE.md` govern it:

1. **Keep the horizontal flip.** Four rotations x {identity, mirror} is uniform measure over
   all 8 elements of D4; dropping the flip leaves C4, an arbitrary subgroup.
2. **`EPOCHS` must rise to 80.** Under uniform D4 only 1/8 of presentations are upright and
   `best_epoch` already averages ~33/40 - at 40 epochs "D4 doesn't help" is confounded with
   "undertrained under D4".
3. **Ablation, not protocol change.** The four reported matrices stay flip-only; these 56 runs
   are the control arm.

In [ ]:
!git status --short results/figures
print("\nThen, from a terminal with push access:")
print(f'  git add results/figures/rotation_sensitivity_{OWNER}.json')
print(f'  git commit -m "Gate 0: rotation sensitivity, {OWNER} half"')
print("  git push")